# Permutation Invariance and Equivariance
One essential property in understanding GNNs is permutation equivariance and invariance, which relates to how these models handle node orderings within a graph.

In this notebook, we'll explore the concept of permutation equivariance or invariance in GNNs using PyTorch Geometric.

Through hands-on examples and explanations, we'll delve into how GNNs maintain consistency in their predictions despite variations in node orderings, providing insights into the robustness and generalization capabilities of these models.

**Run this after the lesson.** It needs the `gnn` package group and nothing
else from this course — no Neo4j, no served model, no Chicago.

**It downloads QM9 the first time: 684 MB** (370 of raw, 314 processed) and
about a hundred seconds, into `data/qm9/`, which is not in git. It is a large
download to use one molecule, and that is the honest trade: the unit tests below
want a real graph with edge features, and QM9's first molecule is one. Delete
`data/qm9/raw/` afterwards if you want the 370 MB back.

## The environment

In [1]:
# The versions this notebook runs on. Pinned in `pyproject.toml`
# under the `gnn` extra, so they are the same for everybody.
import torch
import torch_geometric

print("torch", torch.__version__, "| PyG", torch_geometric.__version__)


/private/tmp/claude-501/-Users-giuseppefutia-Desktop-code-kgtp/211e9142-fbf2-4c28-97e3-4461866db636/scratchpad/venv-verifica/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.5.1 | PyG 2.8.0.post1


# Load Dataset
The QM9 dataset, as described in the "MoleculeNet: A Benchmark for Molecular Machine Learning" paper, comprises approximately 130,000 molecules, each associated with 19 regression targets. Each molecule in the dataset contains comprehensive spatial details for its single low-energy conformation. Furthermore, the dataset includes atom features sourced from the "Neural Message Passing for Quantum Chemistry" paper.

In [2]:
from torch_geometric.datasets import QM9
dataset = QM9(root='../../data/qm9/')

Extracting ../../data/qm9/raw/qm9_v3.zip


Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.


Done!


# Import Libraries

In [3]:
import random
import numpy as np

import torch
import torch.nn.functional as F
from torch.nn import Linear, ReLU, BatchNorm1d, Module, Sequential

import torch_geometric
from torch_geometric.data import Data
from torch_geometric.data import Batch
from torch_geometric.datasets import QM9
import torch_geometric.transforms as T
from torch_geometric.utils import remove_self_loops, to_dense_adj, dense_to_sparse
from torch_geometric.loader import DataLoader
from torch_geometric.nn import MessagePassing, global_mean_pool
from torch_geometric.datasets import QM9
# `torch_geometric.utils.scatter` and not `torch_scatter`: the same
# function, accepting the same reduce='add', without a compiled
# extension that has to match the torch build.
from torch_geometric.utils import scatter

# Create the MPNN Layer from scratch

In [4]:
class MPNNLayer(MessagePassing):
    def __init__(self, emb_dim=64, edge_dim=4, aggr='add'):
        """Message Passing Neural Network Layer

        Args:
            emb_dim: (int) - hidden dimension `d`
            edge_dim: (int) - edge feature dimension `d_e`
            aggr: (str) - aggregation function `\oplus` (sum/mean/max)
        """
        # Set the aggregation function
        super().__init__(aggr=aggr)

        self.emb_dim = emb_dim
        self.edge_dim = edge_dim

        # MLP `\psi` for computing messages `m_ij`
        # Implemented as a stack of Linear->BN->ReLU->Linear->BN->ReLU
        # dims: (2d + d_e) -> d
        self.mlp_msg = Sequential(
            Linear(2*emb_dim + edge_dim, emb_dim), BatchNorm1d(emb_dim), ReLU(),
            Linear(emb_dim, emb_dim), BatchNorm1d(emb_dim), ReLU()
          )

        # MLP `\phi` for computing updated node features `h_i^{l+1}`
        # Implemented as a stack of Linear->BN->ReLU->Linear->BN->ReLU
        # dims: 2d -> d
        self.mlp_upd = Sequential(
            Linear(2*emb_dim, emb_dim), BatchNorm1d(emb_dim), ReLU(),
            Linear(emb_dim, emb_dim), BatchNorm1d(emb_dim), ReLU()
          )

    def forward(self, h, edge_index, edge_attr):
        """
        `propagate()` function starts the message passing procedure:
        `message()` -> `aggregate()` -> `update()`.
        """
        out = self.propagate(edge_index, h=h, edge_attr=edge_attr)
        return out

    def message(self, h_i, h_j, edge_attr):
        msg = torch.cat([h_i, h_j, edge_attr], dim=-1)
        return self.mlp_msg(msg)

    def aggregate(self, inputs, index):
        return scatter(inputs, index, dim=self.node_dim, reduce=self.aggr)

    def update(self, aggr_out, h):
        upd_out = torch.cat([h, aggr_out], dim=-1)
        return self.mlp_upd(upd_out)

    def __repr__(self) -> str:
        return (f'{self.__class__.__name__}(emb_dim={self.emb_dim}, aggr={self.aggr})')

# Create the MPNN Model from the Created Layer

In [5]:
class MPNNModel(Module):
    def __init__(self, num_layers=4, emb_dim=64, in_dim=11, edge_dim=4, out_dim=1):
        """Message Passing Neural Network model for graph property prediction

        Args:
            num_layers: (int) - number of message passing layers `L`
            emb_dim: (int) - hidden dimension `d`
            in_dim: (int) - initial node feature dimension `d_n`
            edge_dim: (int) - edge feature dimension `d_e`
            out_dim: (int) - output dimension (fixed to 1)
        """
        super().__init__()

        # Linear projection for initial node features
        # dim: d_n -> d
        self.lin_in = Linear(in_dim, emb_dim)

        # Stack of MPNN layers
        self.convs = torch.nn.ModuleList()
        for layer in range(num_layers):
            self.convs.append(MPNNLayer(emb_dim, edge_dim, aggr='add'))

        # Global pooling/readout function `R` (mean pooling)
        # PyG handles the underlying logic via `global_mean_pool()`
        self.pool = global_mean_pool

        # Linear prediction head
        # dim: d -> out_dim
        self.lin_pred = Linear(emb_dim, out_dim)

    def forward(self, data):
        """
        Args:
            data: (PyG.Data) - batch of PyG graphs

        Returns:
            out: (batch_size, out_dim) - prediction for each graph
        """
        h = self.lin_in(data.x) # (n, d_n) -> (n, d)

        for conv in self.convs:
            h = h + conv(h, data.edge_index, data.edge_attr) # (n, d) -> (n, d)
            # Note that we add a residual connection after each MPNN layer

        h_graph = self.pool(h, data.batch) # (n, d) -> (batch_size, d)

        out = self.lin_pred(h_graph) # (batch_size, d) -> (batch_size, 1)

        return out.view(-1)

# Permutation test functions

Permutation equivariance at the layer level means that the output of a single GNN layer remains consistent even if the order of the input nodes is changed. This implies that if we shuffle the nodes in the input graph, the output of the GNN layer will also be shuffled in the same way.

Permutation invariance at the model level means that the overall output of the entire GNN model remains unchanged regardless of permutations in the input data. This means that if we reorder the nodes in the input graph, the final output of the GNN model will be the same.

In [6]:
def permute_graph(data, perm):
    """Helper function for permuting PyG Data object attributes consistently.
    """
    # Permute the node attribute ordering
    data.x = data.x[perm]
    data.pos = data.pos[perm]
    data.z = data.z[perm]
    # data.batch = data.batch[perm]

    # Permute the edge index
    adj = to_dense_adj(data.edge_index)
    adj = adj[:, perm, :]
    adj = adj[:, :, perm]
    data.edge_index = dense_to_sparse(adj)[0]
    return data

def permutation_invariance_unit_test(module, dataloader):
    """Unit test for checking whether a module (GNN model) is
    permutation invariant.
    """
    it = iter(dataloader)
    data = next(it)

    # Set edge_attr to dummy values (for simplicity)
    data.edge_attr = torch.zeros(data.edge_attr.shape)

    # Forward pass on original example
    out_1 = module(data)
    print("--- Input graph BEFORE the permutation")
    print("Input data", data)
    print("\nEdge index", data.edge_index)
    print("\nNode features", data.x)
    print("\nModule output", out_1)

    # Create random permutation
    torch.manual_seed(0)
    perm = torch.randperm(data.x.shape[0])
    print("\n\n---Permutation", perm)
    data = permute_graph(data, perm)

    # Forward pass on permuted example
    out_2 = module(data)
    print("\n\n--- Input graph AFTER the permutation")
    print("Input data", data)
    print("\nEdge index", data.edge_index)
    print("\nNode features", data.x)
    print("\nModule output", out_2)

    # Check whether output varies after applying transformations
    return torch.allclose(out_1, out_2, atol=1e-04)


def permutation_equivariance_unit_test(module, dataloader):
    """Unit test for checking whether a module (GNN layer) is
    permutation equivariant.
    """
    it = iter(dataloader)
    data = next(it)

    # Set edge_attr to dummy values (for simplicity)
    data.edge_attr = torch.zeros(data.edge_attr.shape)

    print("--- Input graph BEFORE the permutation")
    print("Input data", data)
    print("\nEdge index", data.edge_index)
    print("\nNode features", data.x)

    # Forward pass on original example
    out_1 = module(data.x, data.edge_index, data.edge_attr)

    # Create random permutation
    torch.manual_seed(0)
    perm = torch.randperm(data.x.shape[0])
    data = permute_graph(data, perm)

    print("\n\n--- Input graph AFTER the permutation")
    print("Input data", data)
    print("\nEdge index", data.edge_index)
    print("\nNode features", data.x)

    # Forward pass on permuted example
    out_2 = module(data.x, data.edge_index, data.edge_attr)
    print('\n\n--- Output of the MP layer before the permutation', out_1)
    print('\n--- Output of the MP layer after the permutation', out_2)
    print('\n--- Applying the permutation to output of the MP layer before the permutation', out_1[perm])

    # Check whether output varies after applying transformations
    return torch.allclose(out_1[perm], out_2, atol=1e-04)

# Perform invariant and equivariant unit test

In [7]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

In [8]:
# Permutation invariance unit test for MPNN model
model = MPNNModel(num_layers=4, emb_dim=64, in_dim=11, edge_dim=4, out_dim=1)
print(f"\n\n####Is {type(model).__name__} permutation invariant? #### --> {permutation_invariance_unit_test(model, dataloader)}!")

--- Input graph BEFORE the permutation
Input data DataBatch(x=[5, 11], edge_index=[2, 8], edge_attr=[8, 4], y=[1, 19], pos=[5, 3], idx=[1], name=[1], z=[5], batch=[5], ptr=[2])

Edge index tensor([[0, 0, 0, 0, 1, 2, 3, 4],
        [1, 2, 3, 4, 0, 0, 0, 0]])

Node features tensor([[0., 1., 0., 0., 0., 6., 0., 0., 0., 0., 4.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.]])

Module output tensor([-0.0573], grad_fn=<ViewBackward0>)


---Permutation tensor([4, 0, 1, 3, 2])


--- Input graph AFTER the permutation
Input data DataBatch(x=[5, 11], edge_index=[2, 8], edge_attr=[8, 4], y=[1, 19], pos=[5, 3], idx=[1], name=[1], z=[5], batch=[5], ptr=[2])

Edge index tensor([[0, 1, 1, 1, 1, 2, 3, 4],
        [1, 0, 2, 3, 4, 1, 1, 1]])

Node features tensor([[1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 

In [9]:
# Instantiate temporary model, layer, and dataloader for unit testing
layer = MPNNLayer(emb_dim=11, edge_dim=4)
# Permutation equivariance unit for MPNN layer
print(f"\n\nIs {type(layer).__name__} permutation equivariant? --> {permutation_equivariance_unit_test(layer, dataloader)}!")

--- Input graph BEFORE the permutation
Input data DataBatch(x=[5, 11], edge_index=[2, 8], edge_attr=[8, 4], y=[1, 19], pos=[5, 3], idx=[1], name=[1], z=[5], batch=[5], ptr=[2])

Edge index tensor([[0, 0, 0, 0, 1, 2, 3, 4],
        [1, 2, 3, 4, 0, 0, 0, 0]])

Node features tensor([[0., 1., 0., 0., 0., 6., 0., 0., 0., 0., 4.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.]])


--- Input graph AFTER the permutation
Input data DataBatch(x=[5, 11], edge_index=[2, 8], edge_attr=[8, 4], y=[1, 19], pos=[5, 3], idx=[1], name=[1], z=[5], batch=[5], ptr=[2])

Edge index tensor([[0, 1, 1, 1, 1, 2, 3, 4],
        [1, 0, 2, 3, 4, 1, 1, 1]])

Node features tensor([[1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 6., 0., 0., 0., 0., 4.],
        [1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [1., 0., 0.,